In [0]:
1 == 1

# Build Waves Setup Helper

Deploy and configure a `genie-build-waves` instance for the **Underwriting Genie** project.

**Run order:** Step 1 → Step 2 (terminal only) → Step 3 → Step 4 → Step 5

> **Note for Genie Code:** Run each numbered cell in order. Read every printed output before moving to the next step. If Step 3 prints any ❌ errors, fix `docs/plans/waves.json` and re-run Step 3 before proceeding.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
# ============================================================
# PROJECT CONFIGURATION — edit these for each new project
# ============================================================
PROJECT_NAME       = "uw-genie"
INSTANCE_SUFFIX    = "-uw-genie"
BUILD_ROOT         = "/Workspace/Users/matthew.giglia@databricks.com/underwritingGenie"
BUILD_BRANCH       = "mg-genie-uw-genie-build-1"
MANIFEST_PATH      = f"{BUILD_ROOT}/docs/plans/waves.json"
SKILLS_CATALOG     = "hls_fde_dev"
SKILLS_SCHEMA      = "dev_matthew_giglia_build_waves"
CREATE_SKILLS_SCHEMA = "false"
MONITOR_PRINCIPAL  = "72bab3e9-303d-4443-b43e-9eaab474d8bb"
MONITOR_JOB_LEVEL  = "CAN_MANAGE_RUN"
BUNDLE_DIR         = "/Workspace/Users/matthew.giglia@databricks.com/mbp/genie-build-waves"
BUNDLE_TARGET      = "dev"

# Resolved at runtime — do not edit
from databricks.sdk import WorkspaceClient
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_host = _ctx.apiUrl().get()
_token = _ctx.apiToken().get()
if not _host.startswith("https://"):
    _host = f"https://{_host}"

w = WorkspaceClient(host=_host, token=_token)
print(f"Workspace : {_host}")
print(f"Project   : {PROJECT_NAME}  (suffix {INSTANCE_SUFFIX})")
print(f"Branch    : {BUILD_BRANCH}")
print(f"Manifest  : {MANIFEST_PATH}")
print(f"Skills UC : {SKILLS_CATALOG}.{SKILLS_SCHEMA}")

In [0]:
import json, os

ERRORS = []
WARNS  = []

# --- Load manifest ---
if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

rt_keys = set(manifest.get("runtime_targets", {}).keys())
print(f"Runtime targets defined: {sorted(rt_keys) or '(none)'}")

runnable = [w for w in manifest.get("waves", []) if not w.get("archived")]
print(f"Waves: {len(manifest['waves'])} total, {len(runnable)} runnable\n")

for wave in runnable:
    wid = wave["wave_id"]
    rg  = set(wave.get("runtime_gate", []))
    for session in wave.get("sessions", []):
        sid  = session["session_id"]
        mode = session.get("mode", "edit")
        srt  = session.get("runtime_targets")
        where = f"wave {wid} / session {sid}"

        if mode == "converge":
            # Rule 1: runtime_targets must exist and be non-empty
            if srt is None:
                ERRORS.append(f"{where}: converge session is missing 'runtime_targets'")
            elif not srt:  # empty list — `not []` is True
                ERRORS.append(f"{where}: converge session has runtime_targets: [] (must be non-empty)")
            else:
                # Rule 2: every key must also appear in the wave's runtime_gate
                for key in srt:
                    if key not in rt_keys:
                        ERRORS.append(f"{where}: runtime_targets key '{key}' not in top-level runtime_targets")
                    if key not in rg:
                        ERRORS.append(f"{where}: runtime_targets key '{key}' must also be in wave {wid} runtime_gate")
        else:
            if srt is not None:
                WARNS.append(f"{where}: non-converge session has runtime_targets (unexpected, will be ignored)")

if ERRORS:
    for e in ERRORS:
        print(f"  \u274c {e}")
    raise ValueError(f"waves.json has {len(ERRORS)} error(s) — fix before continuing")
elif WARNS:
    for w_msg in WARNS:
        print(f"  \u26a0\ufe0f  {w_msg}")
    print("\n\u2705 Manifest valid (with warnings)")
else:
    print("\u2705 Manifest valid — no errors")

In [0]:
# The Databricks CLI cannot run inside a serverless notebook.
# Copy-paste this command into the web terminal (or a cluster notebook).
# Run deploy_instance.py FIRST, then proceed to Step 4 (setup job).

import shlex

vars_map = {
    "project_name":        PROJECT_NAME,
    "instance_suffix":     INSTANCE_SUFFIX,
    "build_root":          BUILD_ROOT,
    "build_branch":        BUILD_BRANCH,
    "manifest_path":       MANIFEST_PATH,
    "skills_catalog":      SKILLS_CATALOG,
    "skills_schema":       SKILLS_SCHEMA,
    "create_skills_schema": CREATE_SKILLS_SCHEMA,
    "monitor_principal":   MONITOR_PRINCIPAL,
    "monitor_job_level":   MONITOR_JOB_LEVEL,
}
vars_flags = " ".join(f'--var={k}={v}' for k, v in vars_map.items())
deploy_cmd = (
    f"cd {BUNDLE_DIR}\n"
    f"python scripts/deploy_instance.py --target {BUNDLE_TARGET} "
    f"--vars {BUILD_ROOT}/docs/plans/build-waves.vars.json"
)
print("=" * 70)
print("TERMINAL COMMAND — run this BEFORE Step 4:")
print("=" * 70)
print(deploy_cmd)
print()
print("(build-waves.vars.json already contains all --var values)")

In [0]:
# Looks up deployed jobs by name pattern. Run after `bundle deploy` succeeds.
# If no jobs are found, run Step 3 first (bundle deploy in terminal).

from databricks.sdk.service.jobs import BaseJob

pattern = PROJECT_NAME  # e.g. 'uw-genie'
matches = [
    j for j in w.jobs.list(name=pattern)
    if j.settings and j.settings.name and pattern.lower() in j.settings.name.lower()
]

if not matches:
    print(f"\u274c No jobs found matching '{pattern}'. Run the bundle deploy in the terminal first.")
else:
    print(f"\u2705 Found {len(matches)} deployed job(s):\n")
    job_ids = {}
    for j in matches:
        name = j.settings.name
        jid  = j.job_id
        job_ids[name] = jid
        print(f"  {name}")
        print(f"    job_id : {jid}")
        print(f"    URL    : {_host}/jobs/{jid}")
        print()

    # Surface the three canonical IDs
    build_job_id = next((v for k, v in job_ids.items() if "Build Waves" in k and "Setup" not in k and "Smoke" not in k), None)
    setup_job_id = next((v for k, v in job_ids.items() if "Setup" in k), None)
    smoke_job_id = next((v for k, v in job_ids.items() if "Smoke" in k), None)
    print(f"  build_waves     job_id = {build_job_id}")
    print(f"  setup_build_waves job_id = {setup_job_id}")
    print(f"  smoke job_id            = {smoke_job_id}")

In [0]:
# Triggers the setup job (which patches the build job task graph, publishes skills,
# creates automations, and grants the monitor SPN).
# Uses the REST API directly — works on serverless without the CLI.
import time, requests

if not setup_job_id:
    raise RuntimeError("setup_job_id is not set. Run Step 4 first.")

print(f"Triggering setup job {setup_job_id} ...")
run = w.jobs.run_now(job_id=setup_job_id)
run_id = run.run_id
print(f"Run ID : {run_id}")
print(f"URL    : {_host}/jobs/{setup_job_id}/runs/{run_id}\n")

headers = {"Authorization": f"Bearer {_token}"}
for _ in range(60):
    r = requests.get(f"{_host}/api/2.1/jobs/runs/get?run_id={run_id}", headers=headers).json()
    state  = r["state"]["life_cycle_state"]
    result = r["state"].get("result_state", "")
    print(f"  {state} {result}")
    if state in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
        # Extract notebook output
        for t in r.get("tasks", []):
            tid = t.get("run_id")
            if tid:
                out = requests.get(f"{_host}/api/2.1/jobs/runs/get-output?run_id={tid}", headers=headers).json()
                err = out.get("error", "")
                nb  = out.get("notebook_output", {}).get("result", "")
                if err:
                    print(f"\n\u274c Setup FAILED:\n{err[:800]}")
                elif nb:
                    setup_result = json.loads(nb)
                    print("\n\u2705 Setup succeeded. Outputs:")
                    print(json.dumps(setup_result, indent=2))
        break
    time.sleep(10)

In [0]:
# Summarise all IDs needed for:
#   - docs/plans/build-0/devin-monitor-prompt.md
#   - PROJECT_MEMORY.md
#   - Session summary docs

try:
    sr = setup_result
except NameError:
    print("\u26a0\ufe0f  setup_result not set — run Step 5 first")
    sr = {}

print("=" * 60)
print("KEY IDs — copy into docs / monitor prompt")
print("=" * 60)
print(f"build_waves job_id      : {sr.get('build_job_id',  build_job_id)}")
print(f"smoke job_id            : {sr.get('smoke_job_id',  smoke_job_id)}")
print(f"setup job_id            : {setup_job_id}")
print()
for skill_key, fqn in sr.get("skills_published", {}).items():
    print(f"  skill ({skill_key:12s})  : {fqn}")
print()
for cfg_key, cfg_id in sr.get("configuration_ids", {}).items():
    print(f"  automation ({cfg_key:10s}) : {cfg_id}")
print()
print("Monitor grants:")
for g in sr.get("monitor_grants", {}).get("already_held", []):
    print(f"  \u2713  {g}")
for g in sr.get("monitor_grants", {}).get("granted", []):
    print(f"  + {g}")
for g in sr.get("monitor_grants", {}).get("failed", []):
    print(f"  \u274c FAILED: {g}")
print()
print(f"Job URL (build) : {_host}/jobs/{sr.get('build_job_id', build_job_id)}")
print(f"Job URL (setup) : {_host}/jobs/{setup_job_id}")

In [0]:
# Run this cell only after Step 5 succeeded (setup job passed).
# Triggers the first build run. Monitor it via the Job URL printed above.
#
# NOTE FOR GENIE CODE: Only run this cell when explicitly instructed.
# Check that the setup job succeeded (Step 5 printed ✅) before triggering.

TRIGGER_BUILD = False  # Set to True to trigger

if not TRIGGER_BUILD:
    print("Skipped — set TRIGGER_BUILD = True to start the build run.")
elif not build_job_id:
    print("\u274c build_job_id not set. Run Step 4 first.")
else:
    build_run = w.jobs.run_now(job_id=build_job_id)
    print(f"\u2705 Build run triggered")
    print(f"  Run ID : {build_run.run_id}")
    print(f"  URL    : {_host}/jobs/{build_job_id}/runs/{build_run.run_id}")